In [53]:
import pandas as pd
import os
import sys

import matplotlib.pyplot as plt

current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

try:
    import data_utils
    from base_utils_qwen import prepare_bayesian_space, competition_scorer as bfrb_competition_scorer, evaluate_holdout, SequenceExtractor
    from proto_utils_v4 import V4PrototypicalNetwork, V4MultiHeadPrototypicalNetwork
    print("✅ Imports loaded successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

✅ Imports loaded successfully


In [ ]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)


✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data


In [12]:
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

In [40]:
child_rows = train_demo_df['age']  <= 18

train_demo_df.loc[child_rows, 'age_band'] = 'child'
train_demo_df.loc[(train_demo_df['age'] > 18) & (train_demo_df['age'] < 25), 'age_band'] = 'young_adult'
train_demo_df.loc[train_demo_df['age'] >= 25, 'age_band'] = 'adult'

train_demo_df.groupby(['handedness', 'age_band']).agg(**{
    'subjects':('subject','unique'),
    'subject count':('subject', 'nunique')})

subjects  \
handedness age_band                                                         
0          adult        [SUBJ_002923, SUBJ_013623, SUBJ_019756, SUBJ_0...   
           child        [SUBJ_028998, SUBJ_039234, SUBJ_055211, SUBJ_0...   
           young_adult                         [SUBJ_032233, SUBJ_041243]   
1          adult        [SUBJ_000206, SUBJ_003328, SUBJ_011323, SUBJ_0...   
           child        [SUBJ_001430, SUBJ_004117, SUBJ_008304, SUBJ_0...   
           young_adult  [SUBJ_012088, SUBJ_019297, SUBJ_020948, SUBJ_0...   

                        subject count  
handedness age_band                    
0          adult                    4  
           child                    4  
           young_adult              2  
1          adult                   26  
           child                   35  
           young_adult             10

In [14]:
train_df.groupby(['bfrb']).agg(**{
    'Sequence Count':('sequence_id','nunique')
})

,Sequence Count
bfrb,
Above ear - pull hair,638
Cheek - pinch skin,637
Eyebrow - pull hair,638
Eyelash - pull hair,640
Forehead - pull hairline,640
Forehead - scratch,640
Neck - pinch skin,640
Neck - scratch,640
non_bfrb,3038


In [78]:
adult_right_subject = 'SUBJ_012088'
adult_right_sequence_id = 'SEQ_000063'

adult_right_sequence_df = train_df[train_df['subject'] == adult_right_subject]
adult_right_sequence_df = adult_right_sequence_df[adult_right_sequence_df['sequence_id'] == adult_right_sequence_id]
adult_right_sequence_df.head()

,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,acc_y,...,tof_5_v58,tof_5_v59,tof_5_v60,tof_5_v61,tof_5_v62,tof_5_v63,gesture_position,gesture_action,is_target,bfrb
row_id,,,,,,,,,,,,,,,,,,,,,
SEQ_000063_000000,Target,SEQ_000063,0,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.363281,-4.378906,...,230.0,238.0,230.0,219.0,214.0,212.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000001,Target,SEQ_000063,1,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.402344,-4.378906,...,228.0,238.0,236.0,224.0,214.0,216.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000002,Target,SEQ_000063,2,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.324219,-4.417969,...,238.0,235.0,234.0,221.0,215.0,214.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000003,Target,SEQ_000063,3,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.359375,-4.378906,...,230.0,237.0,233.0,224.0,215.0,213.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000004,Target,SEQ_000063,4,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.398438,-4.378906,...,226.0,239.0,235.0,222.0,218.0,213.0,Eyelash,pull hair,True,Eyelash - pull hair


In [98]:
sequence_extractor = SequenceExtractor(
    acc_modes = 'raw'
)

transformed = sequence_extractor.fit_transform(adult_right_sequence_df)

In [99]:
transformed

{'X': array([[[ 7.3632812e+00, -4.3789062e+00,  5.2500000e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         [ 7.4023438e+00, -4.3789062e+00,  5.3632812e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         [ 7.3242188e+00, -4.4179688e+00,  5.3242188e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         ...,
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02],
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02],
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02]]],
       shape=(1, 128, 111), dtype=float32),
 'sequence_ids': array(['SEQ_000063'], dtype='<U10')}

In [82]:
transformed.keys()

dict_keys(['X', 'sequence_ids'])